In [1]:
#IMPORTING LIBRARIES
import pandas as pd
import numpy as np
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,ConfusionMatrixDisplay,classification_report
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import randint
from sklearn.metrics import roc_curve, roc_auc_score
from sklearn.metrics import precision_score, recall_score, f1_score
from matplotlib import pyplot as plt
import joblib
from preprocessing import preprocess





In [4]:
# train.py
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# 1. Load data
df = pd.read_csv("train.csv")

# 2. Prepare features/target
X = df.drop(columns=["Churn"])
y = df["Churn"].map({"Yes": 1, "No": 0})

# 3. Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Define preprocessing
categorical_cols = [
    "gender", "Partner", "Dependents", "PhoneService", "MultipleLines",
    "InternetService", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies", "Contract",
    "PaperlessBilling", "PaymentMethod"
]

numeric_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numeric_cols)
    ],
    remainder="drop"
)

# 5. Build pipeline
model = Pipeline([
    ("preprocess", preprocess),
    ("rf", RandomForestClassifier(
        random_state=42,
        class_weight="balanced",
        n_estimators=30,
        max_depth=10,
        min_samples_split=4,
        min_samples_leaf=2,
        n_jobs=-1
    ))
])

# 6. Train
model.fit(X_train, y_train)

# 7. Evaluate
y_pred = model.predict(X_test)
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# 8. Save pipeline
joblib.dump(model, "customer_churn_pipeline.pkl")
print("Saved model as customer_churn_pipeline.pkl")

Precision: 0.5462555066079295
Recall: 0.884953106901319
F1-score: 0.6755276668568169
              precision    recall  f1-score   support

           0       0.96      0.79      0.86     92076
           1       0.55      0.88      0.68     26763

    accuracy                           0.81    118839
   macro avg       0.75      0.84      0.77    118839
weighted avg       0.87      0.81      0.82    118839

Saved model as customer_churn_pipeline.pkl
